# 02 - Outlier Profiling & Cleaning Rules Evaluation

This notebook tests data hygiene thresholds and edge cases on the interim venue data:
- Exact duplicate trips
- Distribution percentiles (1st, 5th, 50th, 95th, 99th, 99.9th)
- Unreasonable trip distances and durations
- Non-positive or extreme surge passenger fares

In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path('../data/interim/venue_trips_2024-01.parquet')
if not DATA_PATH.exists():
    DATA_PATH = Path('data/interim/venue_trips_2024-01.parquet')

df = pd.read_parquet(DATA_PATH)
print(f'Interim baseline records loaded: {len(df):,}')

### 1. Duplicate Trip Analysis

In [ ]:
n_duplicates = df.duplicated().sum()
print(f'Total exact duplicate rows: {n_duplicates:,}')

### 2. Numerical Summary Statistics & Percentile Bounds

In [ ]:
metrics = ['trip_miles', 'trip_time', 'base_passenger_fare', 'driver_pay']
percentiles = [0.001, 0.01, 0.05, 0.50, 0.95, 0.99, 0.999]
df[metrics].describe(percentiles=percentiles).round(2)

### 3. Edge Case & Anomaly Auditing
Quantifying observations that violate physical or economic sanity bounds:

In [ ]:
anomalies = {
    'Zero or Negative Miles (<= 0)': (df['trip_miles'] <= 0).sum(),
    'Excessive Miles (> 100 mi)': (df['trip_miles'] > 100).sum(),
    'Sub-Minute Trips (< 60s)': (df['trip_time'] < 60).sum(),
    'Trips Over 4 Hours (> 14,400s)': (df['trip_time'] > 14400).sum(),
    'Negative or Zero Base Fare (<= $0)': (df['base_passenger_fare'] <= 0).sum(),
    'Outlier Base Fare (> $300)': (df['base_passenger_fare'] > 300).sum()
}

anomaly_df = pd.DataFrame(list(anomalies.items()), columns=['Anomaly Description', 'Record Count'])
anomaly_df['Pct of Total (%)'] = (anomaly_df['Record Count'] / len(df) * 100).round(3)
anomaly_df

### 4. Cleaning Rule Dry Run
Testing the filtering rules intended for `src/clean.py`.

In [ ]:
# Apply proposed pipeline filters
clean_mask = (
    (df['trip_miles'] > 0) &
    (df['trip_miles'] <= 100) &
    (df['trip_time'] >= 60) &
    (df['trip_time'] <= 14400) &
    (df['base_passenger_fare'] > 0)
)

cleaned_df = df[clean_mask].drop_duplicates()

removed_rows = len(df) - len(cleaned_df)
print(f'Original Records: {len(df):,}')
print(f'Cleaned Records:  {len(cleaned_df):,}')
print(f'Rows Dropped:     {removed_rows:,} ({removed_rows / len(df) * 100:.2f}%)')